# Fake vs Real News Classification with Deep Learning

A binary text classification project that uses a neural network to distinguish fake news articles from real ones.

**Dataset:** [Fake and Real News Dataset](https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset) from Kaggle

| File | Description |
|------|-------------|
| `True.csv` | 21,417 real news articles |
| `Fake.csv` | 23,481 fake news articles |

## What You'll Learn

1. **Text preprocessing** - Cleaning, stemming, and stopword removal using NLTK
2. **Feature extraction** - Converting text to numerical features with Bag-of-Words (CountVectorizer)
3. **Neural network** - Building a binary classifier with TensorFlow/Keras
4. **Evaluation** - Accuracy, precision, recall, F1-score, and confusion matrix

---
## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from string import punctuation

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

import nltk
nltk.download('stopwords', quiet=True)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow version: {tf.__version__}")

---
## Step 2: Load Data

Download the dataset from [Kaggle](https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset) and place `True.csv` and `Fake.csv` in a `data/` folder.

In [ ]:
real = pd.read_csv("data/True.csv")
fake = pd.read_csv("data/Fake.csv")

print(f"Real news articles: {len(real):,}")
print(f"Fake news articles: {len(fake):,}")

---
## Step 3: Explore the Data

In [ ]:
print("=== Real News (sample) ===")
display(real.head(3))
print(f"\nColumns: {list(real.columns)}")
print(f"Null values: {real.isna().sum().sum()}")

print("\n=== Fake News (sample) ===")
display(fake.head(3))
print(f"\nNull values: {fake.isna().sum().sum()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

real['subject'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Real News - Subject Distribution')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

fake['subject'].value_counts().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Fake News - Subject Distribution')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## Step 4: Preprocess the Data

**Steps:**
1. Label the data (real=1, fake=0)
2. Combine both datasets
3. Merge text columns into a single feature
4. Apply stemming and remove stopwords

In [ ]:
real['label'] = 1
fake['label'] = 0

df = pd.concat([real, fake], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Combined dataset: {len(df):,} articles")
print(f"\nClass distribution:\n{df['label'].value_counts().rename({1: 'Real', 0: 'Fake'})}")

In [ ]:
df['text'] = df['title'] + " " + df['text'] + " " + df['subject']
df = df[['text', 'label']]

df.head()

### Text Cleaning: Stemming & Stopword Removal

- **Stopwords** are common words ("the", "is", "at") that don't carry meaning for classification
- **Stemming** reduces words to their root form ("running" -> "run")

In [ ]:
stop = set(stopwords.words('english'))
stop.update(punctuation)
stemmer = PorterStemmer()

def clean_text(text):
    tokens = []
    for word in text.split():
        word = word.strip().lower()
        if word not in stop:
            tokens.append(stemmer.stem(word))
    return " ".join(tokens)

print("Before:", df['text'].iloc[0][:200])
df['text'] = df['text'].apply(clean_text)
print("\nAfter: ", df['text'].iloc[0][:200])

---
## Step 5: Feature Extraction (Bag-of-Words)

CountVectorizer converts text into a matrix of token counts. Each unique word becomes a feature column.

- `max_features=50000` limits vocabulary size to keep training feasible
- `ngram_range=(1,2)` captures single words and two-word phrases

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.25, random_state=SEED, stratify=df['label']
)

print(f"Training set: {len(X_train):,} articles")
print(f"Test set:     {len(X_test):,} articles")

In [ ]:
vectorizer = CountVectorizer(max_features=50000, ngram_range=(1, 2))

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Vocabulary size: {len(vectorizer.vocabulary_):,}")
print(f"Training matrix shape: {X_train_vec.shape}")
print(f"Test matrix shape:     {X_test_vec.shape}")

---
## Step 6: Build the Neural Network

A feedforward neural network with:
- Decreasing layer sizes (128 -> 64 -> 32) to compress learned features
- ReLU activation for hidden layers
- Dropout layers to prevent overfitting
- Sigmoid output for binary classification

In [ ]:
model = Sequential([
    Dense(128, activation='relu', input_dim=X_train_vec.shape[1]),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

---
## Step 7: Train the Model

In [ ]:
history = model.fit(
    X_train_vec, y_train,
    epochs=5,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Train Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[1].set_title('Accuracy over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Step 8: Evaluate the Model

In [ ]:
y_pred_prob = model.predict(X_test_vec)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

In [ ]:
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Fake', 'Real']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Fake', 'Real'],
    yticklabels=['Fake', 'Real']
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

---
## Key Takeaways

- **Bag-of-Words + a simple neural network** achieves ~90%+ accuracy on this dataset
- **Stemming and stopword removal** reduce noise and vocabulary size significantly
- **Capping `max_features`** prevents memory issues and speeds up training without losing much signal
- **Dropout** helps reduce overfitting in the neural network

### Ideas to Explore Further

- Try **TF-IDF** instead of raw counts (`TfidfVectorizer`)
- Use **word embeddings** (Word2Vec, GloVe) instead of BoW
- Replace the dense network with an **LSTM or Transformer** model
- Add **early stopping** to prevent overfitting
- Experiment with **different hyperparameters** (learning rate, layer sizes, epochs)